# Notebook Overview (Single Ticker: Benzinga → FinBERT → Time-Series Merge)

**Purpose:**
This notebook builds a **sentiment-augmented daily time series** for a *single* equity ticker (here: `NKE`). It downloads historical Benzinga news, filters overly long articles, scores each article with FinBERT sentiment probabilities, aggregates sentiment per day, and merges the resulting features into the daily closing-price series.

---

## Configuration and Inputs

### Environment
- The Benzinga API key is loaded from a local `.env` file via `python-dotenv`:
  - `BENZINGA_API_KEY` must be present, otherwise the notebook raises an error.

### Directory Layout
The following directories are created if they do not exist:
- `../data/benzinga/` → raw news JSONL
- `../data/finbert_data/` → FinBERT-scored news JSONL
- `../data/ts_with_sentiment/` → merged daily dataset

### Input Files
- **Price time series (daily):**
  `../data/time_series/nke.csv`
  Required columns: `Date`, `Close`

In [1]:
# -------- Imports --------
import os
import json
from html import unescape
from urllib.parse import unquote
from html.parser import HTMLParser
from datetime import datetime, timedelta
from dotenv import load_dotenv

import pandas as pd
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

from benzinga import news_data
from benzinga.news_data import BadRequestError

In [2]:
# -------- Helpers --------
class HTMLTextExtractor(HTMLParser):
    """
    Extract plain text from HTML while ignoring <script> and <style> content.
    """

    def __init__(self):
        super().__init__()
        self._in_script = False
        self._in_style = False
        self._chunks = []

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag == "script":
            self._in_script = True
        elif tag == "style":
            self._in_style = True

    def handle_endtag(self, tag):
        tag = tag.lower()
        if tag == "script":
            self._in_script = False
        elif tag == "style":
            self._in_style = False

    def handle_data(self, data):
        if not self._in_script and not self._in_style:
            self._chunks.append(data)

    def get_text(self) -> str:
        return "".join(self._chunks)


def html_body_to_plain_text(raw_body: str) -> str:
    """
    Convert a raw HTML body string to cleaned plain text.

    Steps:
    1) URL-decode (%20 -> ' ')
    2) Decode HTML entities (&amp; -> &)
    3) Strip HTML tags, <script>, and <style> content
    4) Normalize whitespace
    """
    if not raw_body:
        return ""

    # URL and HTML entities decoding
    decoded = unquote(unescape(raw_body))

    # Parse HTML and drop script/style
    parser = HTMLTextExtractor()
    parser.feed(decoded)
    text = parser.get_text()

    # Whitespace normalization
    return " ".join(text.split())


def generate_yearly_ranges(start_date_str: str, end_date_str: str):
    """
    Generate yearly date ranges as (start, end) tuples of strings.

    Example:
        2009-01-01 .. 2025-01-31 becomes:
        - 2009-01-01 .. 2009-12-31
        - 2010-01-01 .. 2010-12-31
        ...
        - 2025-01-01 .. 2025-01-31
    """
    start = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    end = datetime.strptime(end_date_str, "%Y-%m-%d").date()

    ranges = []
    current_start = start

    while current_start <= end:
        year_end = datetime(current_start.year, 12, 31).date()
        current_end = min(year_end, end)

        ranges.append(
            (
                current_start.strftime("%Y-%m-%d"),
                current_end.strftime("%Y-%m-%d"),
            )
        )

        current_start = current_end + timedelta(days=1)

    return ranges

In [7]:
# -------- Variables --------
load_dotenv()

api_key = os.getenv("BENZINGA_API_KEY")
if api_key is None:
    raise RuntimeError(
        "BENZINGA_API_KEY not found. "
        "Please set it in your .env file."
    )
max_tokens = 512
SEED = 42

ticker = "NKE"
global_date_from = "2009-01-01"
global_date_to = "2025-01-31"

benzinga_directory_path = "../../data/benzinga"
finbert_directory_path = "../../data/finbert_data"
ts_with_sentiment_dir = "../../data/ts_with_sentiment"

os.makedirs(benzinga_directory_path, exist_ok=True)
os.makedirs(finbert_directory_path, exist_ok=True)
os.makedirs(ts_with_sentiment_dir, exist_ok=True)

benzinga_file_path = os.path.join(benzinga_directory_path, f"{ticker}.jsonl")
finbert_file_path = os.path.join(
    finbert_directory_path, f"{ticker}_finbert_sentiment.jsonl"
)
ts_csv_path = f"../../data/time_series/{ticker.lower()}.csv"
output_csv_path = os.path.join(
    ts_with_sentiment_dir, f"{ticker}_ts_with_sentiment.csv"
)

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Ensure deterministic behavior (safe for inference)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"[INFO] Global random seed set to {SEED}")

[INFO] Global random seed set to 42


In [2]:
# -------- Benzinga Download + Filter/Export --------
finbert_tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")

page_size = 100
page_start = 0
page_max = 400

fin = news_data.News(api_key)

# Statistics
total_articles_raw = 0
total_articles_written = 0
total_articles_skipped_too_long = 0

date_ranges = generate_yearly_ranges(global_date_from, global_date_to)

with open(benzinga_file_path, "w", encoding="utf-8") as f_out:
    for date_from, date_to in date_ranges:
        print(f"Processing ticker {ticker}, range {date_from} to {date_to} ...")
        page = page_start

        while True:
            if page > page_max:
                print(
                    f"  Page limit ({page_max}) reached for {date_from}..{date_to}, "
                    f"stopping this range."
                )
                break

            try:
                articles = fin.news(
                    company_tickers=ticker,
                    date_from=date_from,
                    date_to=date_to,
                    display_output="full",
                    pagesize=page_size,
                    page=page,
                )
            except BadRequestError:
                print(
                    f"  BadRequestError for {ticker}, range {date_from}..{date_to}, "
                    f"page {page}. Stopping this range."
                )
                break

            if not articles:
                print(f"  No more articles in range {date_from}..{date_to} (page {page}).")
                break

            for art in articles:
                total_articles_raw += 1

                raw_body = art.get("body", "") or ""
                clean_body = html_body_to_plain_text(raw_body)

                # Remove quotes so later string handling will be safe
                clean_body = clean_body.replace('"', "").replace("'", "")

                # Token length via FinBERT tokenizer
                token_ids = finbert_tokenizer(
                    clean_body, add_special_tokens=True
                )["input_ids"]
                n_tokens = len(token_ids)

                if n_tokens > max_tokens:
                    total_articles_skipped_too_long += 1
                    continue

                if n_tokens == 0:
                    continue

                record = {
                    "created": art.get("created"),
                    "title": (art.get("title") or "").replace('"', "").replace("'", ""),
                    "body": clean_body,
                }

                f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                total_articles_written += 1

            page += 1

        print(f"Finished range: {date_from} .. {date_to}")

print("\n--- Benzinga Summary ---")
print(f"Ticker: {ticker}")
print(f"Total raw articles:                 {total_articles_raw}")
print(f"Saved articles (<= {max_tokens} tokens): {total_articles_written}")
print(f"Skipped articles (> {max_tokens} tokens): {total_articles_skipped_too_long}")
print(f"JSONL output file:                  {benzinga_file_path}")


C:\Users\juli\anaconda3\envs\BachelorThesis\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juli\.cache\huggingface\hub\models--yiyanghkust--finbert-tone. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Verarbeite Ticker AAPL, Zeitraum 2009-01-01 bis 2009-12-31 ...
2025-11-19 15:21:07 [info     ] Status Code: 200 Endpoint: http://api.benzinga.com/api/v2/news/?token=REMOVED_API_KEY&pageSize=100&page=0&displayOutput=full&dateFrom=2009-01-01&dateTo=2009-12-31&tickers=AAPL
  Zeitraum 2009-01-01..2009-12-31, Page 0 erledigt – bisher 24 geschrieben, 1 wegen > 512 Tokens verworfen.
2025-11-19 15:21:07 [info     ] Status Code: 200 Endpoint: http://api.benzinga.com/api/v2/news/?token=REMOVED_API_KEY&pageSize=100&page=1&displayOutput=full&dateFrom=2009-01-01&dateTo=2009-12-31&tickers=AAPL
  Zeitraum 2009-01-01..2009-12-31, Page 1 erledigt – bisher 49 geschrieben, 1 wegen > 512 Tokens verworfen.
2025-11-19 15:21:07 [info     ] Status Code: 200 Endpoint: http://api.benzinga.com/api/v2/news/?token=REMOVED_API_KEY&pageSize=100&page=2&displayOutput=full&dateFrom=2009-01-01&dateTo=2009-12-31&tickers=AAPL
  Zeitraum 2009-01-01..2009-12-31, Page 2 erledigt – bisher 72 geschrieben, 3 wegen > 512 Tokens 

In [19]:
# -------- FinBERT Inference --------
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
model.eval()

print(f"Loaded FinBERT model: {model_name}")
print(f"Input JSONL:   {benzinga_file_path}")
print(f"Output JSONL:  {finbert_file_path}")

total_in = 0
total_scored = 0
total_ignored = 0

with open(benzinga_file_path, "r", encoding="utf-8") as f_in, \
     open(finbert_file_path, "w", encoding="utf-8") as f_out:

    for line_no, line in enumerate(f_in, start=1):
        line = line.strip()
        if not line:
            continue

        total_in += 1

        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            print(f"Line {line_no}: JSONDecodeError, skipping.")
            continue

        text = record.get("body", "") or ""
        if not text:
            continue

        inputs = tokenizer(text, return_tensors="pt")

        try:
            with torch.no_grad():
                outputs = model(**inputs)

            logits = outputs.logits
            probs_tensor = torch.nn.functional.softmax(logits, dim=-1)[0]

            probs = probs_tensor.tolist()
            pred_id = int(torch.argmax(probs_tensor).item())
            label = config.id2label[pred_id]

            prob_dict = {config.id2label[i]: probs[i] for i in range(len(probs))}

            out_record = {
                "created": record.get("created"),
                "title": record.get("title"),
                "label": label,
                "probs": prob_dict,
            }

            f_out.write(json.dumps(out_record, ensure_ascii=False) + "\n")
            total_scored += 1
        except RuntimeError:
            total_ignored += 1

print("\n--- FinBERT Summary ---")
print(f"Total input lines:         {total_in}")
print(f"Total ignored (errors):    {total_ignored}")
print(f"Total successfully scored: {total_scored}")
print(f"Sentiment JSONL file:      {finbert_file_path}")

ERROR! Session/line number was not unique in database. History logging moved to new session 209



KeyboardInterrupt



In [8]:
# -------- CSV Creation (Time Series + Sentiment) --------

# -----------------------------
# 1. Load time series data (Date + Close only)
# -----------------------------
df_price = pd.read_csv(ts_csv_path)

if "Date" in df_price.columns:
    date_col = "Date"
else:
    raise ValueError("No 'Date' column found in time series CSV.")

if "Close" not in df_price.columns:
    raise ValueError("No 'Close' column found in time series CSV.")

df_price = df_price[[date_col, "Close"]]

df_price[date_col] = pd.to_datetime(df_price[date_col], errors="coerce", utc=True)
df_price[date_col] = df_price[date_col].dt.tz_convert(None)

df_price = df_price.set_index(date_col).sort_index()

print("Price index dtype:", df_price.index.dtype)
print("Number of price rows:", len(df_price))

# -----------------------------
# 2. Load FinBERT JSONL and extract per-news sentiment vectors
# -----------------------------
rows = []

with open(finbert_file_path, "r", encoding="utf-8") as f_in:
    for line_no, line in enumerate(f_in, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            print(f"Line {line_no}: JSONDecodeError in FinBERT JSONL, skipping.")
            continue

        created_raw = rec.get("created")
        probs = rec.get("probs", {})

        if not created_raw or not probs:
            continue

        try:
            created_dt = pd.to_datetime(created_raw, errors="coerce", utc=True)
        except Exception:
            created_dt = pd.NaT

        if pd.isna(created_dt):
            continue

        date_only = created_dt.date()

        pos = float(probs.get("positive", 0.0))
        neg = float(probs.get("negative", 0.0))
        neu = float(probs.get("neutral", 0.0))

        rows.append(
            {
                "date": date_only,
                "pos": pos,
                "neg": neg,
                "neu": neu,
            }
        )

df_sent = pd.DataFrame(rows)

if df_sent.empty:
    raise ValueError("No sentiment data loaded from FinBERT JSONL.")

# -----------------------------
# 3. Aggregate per day (mean probabilities and news count)
# -----------------------------
df_sent["date"] = pd.to_datetime(df_sent["date"])

df_daily = (
    df_sent
    .groupby("date")
    .agg(
        pos_mean=("pos", "mean"),
        neg_mean=("neg", "mean"),
        neut_mean=("neu", "mean"),
        news_count=("pos", "count"),
    )
    .sort_index()
)

print("Sentiment index dtype:", df_daily.index.dtype)
print("Number of sentiment days:", len(df_daily))

# -----------------------------
# 4. Restrict price data to the sentiment date range
# -----------------------------
first_sent_date = df_daily.index.min()
last_sent_date = df_daily.index.max()

print("First sentiment day:", first_sent_date)
print("Last sentiment day: ", last_sent_date)

df_price = df_price[
    (df_price.index >= first_sent_date) &
    (df_price.index <= last_sent_date)
]

df_price.index = df_price.index.normalize()
df_price = df_price.sort_index()

# -----------------------------
# 5. Join sentiment features onto price data
# -----------------------------
df_daily.index = pd.to_datetime(df_daily.index)
df_merged = df_price.join(df_daily, how="left")

print("Number of merged rows (price & sentiment):", len(df_merged))

for col in ["pos_mean", "neg_mean", "neut_mean", "news_count"]:
    if col not in df_merged.columns:
        df_merged[col] = 0.0

df_merged[["pos_mean", "neg_mean", "neut_mean", "news_count"]] = (
    df_merged[["pos_mean", "neg_mean", "neut_mean", "news_count"]].fillna(0.0)
)

# -----------------------------
# 6. Keep only desired columns and save as CSV
# -----------------------------
df_final = df_merged[["Close", "pos_mean", "neut_mean", "neg_mean", "news_count"]]

df_final.to_csv(output_csv_path, index=True)

print("\nFinal time series with sentiment features saved to:")
print(output_csv_path)
print("\nSample head:")
display(df_final.head())

Price index dtype: datetime64[ns]
Number of price rows: 6348
Sentiment index dtype: datetime64[ns]
Number of sentiment days: 4373
First sentiment day: 2009-08-10 00:00:00
Last sentiment day:  2025-01-31 00:00:00
Number of merged rows (price & sentiment): 3894

Final time series with sentiment features saved to:
../data/ts_with_sentiment\AAPL_ts_with_sentiment_old.csv

Sample head:


,Close,pos_mean,neut_mean,neg_mean,news_count
Date,,,,,
2009-08-10,4.957017,0.065965,0.919376,0.014659,2.0
2009-08-11,4.900139,0.000000,0.000000,0.000000,0.0
2009-08-12,4.974774,0.000000,0.000000,0.000000,0.0
2009-08-13,5.068363,0.000000,0.000000,0.000000,0.0
2009-08-14,5.019012,0.000000,0.000000,0.000000,0.0
